# Paso 3 — Limpieza de Datos e Ingenieria de Atributos — FinanceAI

Este notebook continua el trabajo de `eda_financeai.ipynb` y prepara los datos para el modelado:

1. **Limpieza de texto** de las descripciones de transacciones (necesaria para el clasificador de gastos).
2. **Tratamiento de variables numericas** (outliers).
3. **Codificacion** de variables categoricas.
4. **Ingenieria de atributos** para ambos modelos (clasificador de gastos y clasificador de perfil financiero).
5. **Split train/test** y **serializacion de artefactos** (vectorizador, scaler, splits) para que el siguiente notebook de modelado los reutilice sin fugas de informacion.

> Todo ajuste (`fit`) de transformadores (TF-IDF, scaler, encoder) se hace **solo sobre el conjunto de entrenamiento**, y se aplica (`transform`) sobre el de prueba — asi evitamos que informacion del test se filtre al entrenamiento (data leakage).

In [1]:
import os
import re
import json
import unicodedata

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

RANDOM_STATE = 42

## 1. Carga de datos

In [2]:
df_trans = pd.read_csv("dataset_transacciones.csv", parse_dates=["fecha"])
df_usr = pd.read_csv("dataset_usuarios_perfil.csv")

print(f"Transacciones: {df_trans.shape}")
print(f"Usuarios: {df_usr.shape}")

Transacciones: (13633, 5)
Usuarios: (800, 16)


## 2. Limpieza de texto de las descripciones

En el EDA vimos que las descripciones traen ruido tipico de datos bancarios reales: mayusculas, codigos de referencia (`REF819242`), codigos de comercio (`#7367`) y nombres de ciudad (Colombia y Mexico). Construimos una funcion de limpieza que:

1. Convierte a minusculas.
2. Quita tildes/acentos.
3. Elimina codigos de referencia y de comercio.
4. Elimina nombres de ciudad conocidos.
5. Elimina cualquier caracter no alfabetico restante.
6. Normaliza espacios.

In [3]:
CIUDADES_RUIDO = [
    "ciudad de mexico", "bogota", "medellin", "barranquilla", "cartagena",
    "guadalajara", "monterrey", "queretaro", "cucuta", "cali", "tijuana", "puebla",
]

def quitar_tildes(texto):
    texto_norm = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in texto_norm if not unicodedata.combining(c))

def limpiar_descripcion(texto):
    texto = texto.lower()
    texto = quitar_tildes(texto)
    texto = re.sub(r"ref\d+", " ", texto)
    texto = re.sub(r"#\d+", " ", texto)
    for ciudad in CIUDADES_RUIDO:
        texto = texto.replace(ciudad, " ")
    texto = re.sub(r"[^a-z\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

df_trans["descripcion_limpia"] = df_trans["descripcion"].apply(limpiar_descripcion)
df_trans[["descripcion", "descripcion_limpia", "categoria"]].sample(10, random_state=RANDOM_STATE)

,descripcion,descripcion_limpia,categoria
9929,TRANSACCION #8954,transaccion,Servicios
6157,MATRICULA UNAM QUERETARO,matricula unam,Educacion
3289,FARMACIAS DEL AHORRO,farmacias del ahorro,Salud
13326,EXITO #8722,exito,Alimentacion
5479,PEAJE AUTOPISTA REF942742,peaje autopista,Transporte
8528,FACTURA ENERIA ENEL CODENSA REF980076,factura eneria enel codensa,Servicios
2910,AGENCIA DE VIAJES #1189,agencia de viajes,Ocio
12470,OXXO #9230 REF163100,oxxo,Alimentacion
2337,JUMBO,jumbo,Alimentacion
9142,RETIRO #1385,retiro,Alimentacion


In [4]:
vacias = (df_trans["descripcion_limpia"] == "").sum()
print(f"Descripciones que quedaron vacias tras la limpieza: {vacias}")

Descripciones que quedaron vacias tras la limpieza: 0


### Hallazgo importante: colision entre un comercio y una ciudad

"Farmacias Guadalajara" es una cadena real (parte de nuestro catalogo de la categoria Salud), pero "Guadalajara" tambien esta en nuestra lista de ciudades a remover. Nuestra limpieza basada en remocion de texto por lista no distingue entre ambos casos y termina borrando el nombre del comercio.

Es una limitacion real y esperable de la limpieza basada en reglas simples (sin diccionario de comercios). La documentamos en vez de ignorarla:

In [5]:
mask_colision = df_trans["descripcion"].str.contains("Farmacias Guadalajara", case=False, na=False)
n_colision = mask_colision.sum()
print(f"Transacciones de 'Farmacias Guadalajara' afectadas por la colision: {n_colision}")
print(f"Porcentaje del total de transacciones: {n_colision / len(df_trans) * 100:.2f}%")
print()
print(df_trans.loc[mask_colision, ["descripcion", "descripcion_limpia"]].drop_duplicates().head())

Transacciones de 'Farmacias Guadalajara' afectadas por la colision: 125
Porcentaje del total de transacciones: 0.92%

                                   descripcion descripcion_limpia
42   FARMACIAS GUADALAJARA CARTAGENA REF983773          farmacias
205      FARMACIAS GUADALAJARA #7661 REF277004          farmacias
356                FARMACIAS GUADALAJARA #8095          farmacias
515                      FARMACIAS GUADALAJARA          farmacias
537                FARMACIAS GUADALAJARA #8717          farmacias


Como el volumen es bajo, para este MVP dejamos la limpieza tal cual (el termino "farmacias" que queda todavia es un indicio util para el modelo, ya que es exclusivo de la categoria Salud en nuestro catalogo). Como mejora futura, se podria usar un diccionario de comercios conocidos en vez de una lista generica de ciudades para evitar este tipo de colisiones.

## 3. Tratamiento de variables numericas: outliers

In [6]:
def limites_iqr(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

reporte_outliers = []
for categoria, grupo in df_trans.groupby("categoria"):
    lim_inf, lim_sup = limites_iqr(grupo["valor"])
    n_outliers = ((grupo["valor"] < lim_inf) | (grupo["valor"] > lim_sup)).sum()
    reporte_outliers.append({
        "categoria": categoria,
        "limite_inferior": round(lim_inf, 2),
        "limite_superior": round(lim_sup, 2),
        "outliers": int(n_outliers),
        "pct_outliers": round(n_outliers / len(grupo) * 100, 2),
    })

df_reporte_outliers = pd.DataFrame(reporte_outliers).sort_values("outliers", ascending=False)
df_reporte_outliers

,categoria,limite_inferior,limite_superior,outliers,pct_outliers
0,Alimentacion,-117.36,397.99,0,0.0
1,Educacion,-136.48,546.34,0,0.0
2,Ocio,-59.62,194.37,0,0.0
3,Otros,-76.71,264.97,0,0.0
4,Salud,-143.34,455.53,0,0.0
5,Servicios,-57.37,216.35,0,0.0
6,Transporte,-84.90,308.18,0,0.0
7,Vivienda,-142.03,926.49,0,0.0


El dataset sintetico tiene rangos de valores acotados por diseno, asi que se esperan pocos o ningun outlier severo. Aun asi dejamos la funcion de tratamiento lista (capping por IQR dentro de cada categoria), porque en produccion, con datos reales de usuarios, si van a aparecer transacciones anomalas.

In [7]:
def limitar_outliers(df, columna, columna_grupo):
    df = df.copy()
    for categoria, grupo in df.groupby(columna_grupo):
        lim_inf, lim_sup = limites_iqr(grupo[columna])
        idx = grupo.index
        df.loc[idx, columna] = df.loc[idx, columna].clip(lower=lim_inf, upper=lim_sup)
    return df

valores_antes = df_trans["valor"].sum()
df_trans = limitar_outliers(df_trans, "valor", "categoria")
valores_despues = df_trans["valor"].sum()

print(f"Suma total de 'valor' antes del capping: {valores_antes:,.2f}")
print(f"Suma total de 'valor' despues del capping: {valores_despues:,.2f}")
print(f"Diferencia: {valores_antes - valores_despues:,.2f}")

Suma total de 'valor' antes del capping: 1,796,253.06
Suma total de 'valor' despues del capping: 1,796,253.06
Diferencia: 0.00


## 4. Ingenieria de atributos — Clasificador de gastos (texto)

In [8]:
X_texto = df_trans["descripcion_limpia"]
y_categoria = df_trans["categoria"]

X_train_txt, X_test_txt, y_train_cat, y_test_cat = train_test_split(
    X_texto, y_categoria, test_size=0.2, random_state=RANDOM_STATE, stratify=y_categoria,
)
print(f"Train: {X_train_txt.shape[0]} transacciones | Test: {X_test_txt.shape[0]} transacciones")

Train: 10906 transacciones | Test: 2727 transacciones


In [9]:
vectorizador_tfidf = TfidfVectorizer(min_df=2, ngram_range=(1, 2))
X_train_tfidf = vectorizador_tfidf.fit_transform(X_train_txt)
X_test_tfidf = vectorizador_tfidf.transform(X_test_txt)

print(f"Tamano del vocabulario: {len(vectorizador_tfidf.vocabulary_)}")
print(f"Dimension X_train_tfidf: {X_train_tfidf.shape}")
print(f"Dimension X_test_tfidf: {X_test_tfidf.shape}")

Tamano del vocabulario: 391
Dimension X_train_tfidf: (10906, 391)
Dimension X_test_tfidf: (2727, 391)


In [10]:
ejemplo_idx = 0
fila = X_train_tfidf[ejemplo_idx]
terminos = vectorizador_tfidf.get_feature_names_out()
no_cero = fila.nonzero()[1]

print(f"Descripcion limpia: '{X_train_txt.iloc[ejemplo_idx]}'")
print(f"Categoria real: {y_train_cat.iloc[ejemplo_idx]}")
print("Terminos con peso TF-IDF > 0:")
for idx in sorted(no_cero):
    print(f"  {terminos[idx]!r}: {fila[0, idx]:.3f}")

Descripcion limpia: 'recibo de pago'
Categoria real: Alimentacion
Terminos con peso TF-IDF > 0:
  'de': 0.410
  'de pago': 0.488
  'pago': 0.342
  'recibo': 0.488
  'recibo de': 0.488


## 5. Ingenieria de atributos — Clasificador de perfil financiero

Ademas de las columnas ya calculadas en el generador (`gasto_total`, `ratio_gasto_ingreso`, gasto absoluto por categoria), construimos:

- **Proporcion de gasto por categoria** respecto al gasto total (mas comparable entre usuarios con ingresos muy distintos que el valor absoluto).
- **Gasto promedio por transaccion**.
- **Categoria de mayor gasto** (feature categorica adicional, codificada como one-hot).
- **Frecuencia de ahorro codificada** de forma ordinal (Baja=0, Media=1, Alta=2), ya que tiene un orden logico.

In [11]:
columnas_gasto = [c for c in df_usr.columns if c.startswith("gasto_") and c != "gasto_total"]

df_feat = df_usr.copy()

for col in columnas_gasto:
    nombre_categoria = col.replace("gasto_", "")
    df_feat[f"pct_{nombre_categoria}"] = (df_feat[col] / df_feat["gasto_total"]).fillna(0)

df_feat["gasto_promedio_transaccion"] = df_feat["gasto_total"] / df_feat["num_transacciones"]
df_feat["categoria_gasto_dominante"] = (
    df_feat[columnas_gasto].idxmax(axis=1).str.replace("gasto_", "", regex=False)
)

columnas_pct = [f"pct_{c.replace('gasto_', '')}" for c in columnas_gasto]
df_feat[["user_id", "gasto_promedio_transaccion", "categoria_gasto_dominante"] + columnas_pct].head()

,user_id,gasto_promedio_transaccion,categoria_gasto_dominante,pct_alimentacion,pct_transporte,pct_salud,pct_vivienda,pct_educacion,pct_ocio,pct_servicios,pct_otros
0,1,135.966364,alimentacion,0.693059,0.087809,0.189933,0.000000,0.000000,0.000000,0.029198,0.000000
1,2,126.556364,vivienda,0.103224,0.243513,0.000000,0.448553,0.000000,0.095617,0.068227,0.040866
2,3,161.570000,alimentacion,0.364715,0.152951,0.128561,0.115420,0.000000,0.074281,0.049724,0.114349
3,4,106.153333,alimentacion,0.315696,0.052063,0.246070,0.000000,0.000000,0.099918,0.026230,0.260022
4,5,200.698750,vivienda,0.168782,0.091300,0.015284,0.498564,0.146024,0.045388,0.034657,0.000000


In [12]:
orden_ahorro = [["Baja", "Media", "Alta"]]
encoder_ahorro = OrdinalEncoder(categories=orden_ahorro)
df_feat["frecuencia_ahorro_cod"] = encoder_ahorro.fit_transform(df_feat[["frecuencia_ahorro"]])

df_feat = pd.get_dummies(df_feat, columns=["categoria_gasto_dominante"], prefix="dominante", dtype=int)

print("Distribucion de la categoria de gasto dominante:")
print(df_usr.assign(
    dominante=df_usr[columnas_gasto].idxmax(axis=1).str.replace("gasto_", "", regex=False)
)["dominante"].value_counts())

Distribucion de la categoria de gasto dominante:
dominante
alimentacion    370
vivienda        201
transporte      125
salud            44
educacion        29
ocio             15
servicios        11
otros             5
Name: count, dtype: int64


## 6. Split train/test y escalado (clasificador de perfil)

In [13]:
columnas_excluir = ["user_id", "perfil_financiero", "frecuencia_ahorro"]
columnas_features = [c for c in df_feat.columns if c not in columnas_excluir]

X_perfil = df_feat[columnas_features]
y_perfil = df_feat["perfil_financiero"]

X_train_perfil, X_test_perfil, y_train_perfil, y_test_perfil = train_test_split(
    X_perfil, y_perfil, test_size=0.2, random_state=RANDOM_STATE, stratify=y_perfil,
)

print(f"Train: {X_train_perfil.shape} | Test: {X_test_perfil.shape}")
print()
print("Columnas de features:")
print(columnas_features)

Train: (640, 31) | Test: (160, 31)

Columnas de features:
['ingreso_mensual', 'nivel_endeudamiento', 'num_transacciones', 'gasto_total', 'ratio_gasto_ingreso', 'gasto_alimentacion', 'gasto_transporte', 'gasto_salud', 'gasto_vivienda', 'gasto_educacion', 'gasto_ocio', 'gasto_servicios', 'gasto_otros', 'pct_alimentacion', 'pct_transporte', 'pct_salud', 'pct_vivienda', 'pct_educacion', 'pct_ocio', 'pct_servicios', 'pct_otros', 'gasto_promedio_transaccion', 'frecuencia_ahorro_cod', 'dominante_alimentacion', 'dominante_educacion', 'dominante_ocio', 'dominante_otros', 'dominante_salud', 'dominante_servicios', 'dominante_transporte', 'dominante_vivienda']


In [14]:
columnas_numericas_continuas = sorted(set(
    ["ingreso_mensual", "nivel_endeudamiento", "num_transacciones", "gasto_total",
     "ratio_gasto_ingreso", "gasto_promedio_transaccion"]
    + [c for c in columnas_features if c.startswith("gasto_") or c.startswith("pct_")]
))

scaler_perfil = StandardScaler()
X_train_perfil_scaled = X_train_perfil.copy()
X_test_perfil_scaled = X_test_perfil.copy()

X_train_perfil_scaled[columnas_numericas_continuas] = scaler_perfil.fit_transform(
    X_train_perfil[columnas_numericas_continuas]
)
X_test_perfil_scaled[columnas_numericas_continuas] = scaler_perfil.transform(
    X_test_perfil[columnas_numericas_continuas]
)

print("Nota: el escalado es opcional para modelos de arbol (Random Forest / Gradient Boosting),")
print("pero lo dejamos listo por si se prueba Regresion Logistica u otro modelo sensible a escala.")
X_train_perfil_scaled.head()

Nota: el escalado es opcional para modelos de arbol (Random Forest / Gradient Boosting),
pero lo dejamos listo por si se prueba Regresion Logistica u otro modelo sensible a escala.


,ingreso_mensual,nivel_endeudamiento,num_transacciones,gasto_total,ratio_gasto_ingreso,gasto_alimentacion,gasto_transporte,gasto_salud,gasto_vivienda,gasto_educacion,gasto_ocio,gasto_servicios,gasto_otros,pct_alimentacion,pct_transporte,pct_salud,pct_vivienda,pct_educacion,pct_ocio,pct_servicios,pct_otros,gasto_promedio_transaccion,frecuencia_ahorro_cod,dominante_alimentacion,dominante_educacion,dominante_ocio,dominante_otros,dominante_salud,dominante_servicios,dominante_transporte,dominante_vivienda
353,0.572949,-0.080164,-1.652856,-1.748241,-1.246834,-0.522912,-0.941465,-1.055269,-0.960143,-0.666885,-0.874902,-0.244758,-1.047358,1.975036,0.248795,-1.080310,-1.039013,-0.688075,-0.076897,1.462612,-1.009547,-1.250832,1.0,1,0,0,0,0,0,0,0
301,-0.432077,0.490633,0.510760,-0.207846,-0.079767,-0.931618,-0.281379,-1.055269,0.028854,-0.666885,2.491850,1.161687,0.725487,-1.021922,-0.224781,-1.080310,0.249251,-0.688075,2.446556,1.247721,0.731923,-1.038342,2.0,0,0,1,0,0,0,0,0
148,-1.461961,-0.656503,-0.210445,0.325665,2.213544,1.902042,-0.301651,0.559519,-0.542067,1.120487,-1.158329,-1.227779,-1.047358,1.724955,-0.499828,0.373721,-0.590901,1.057802,-1.125619,-1.216855,-1.009547,1.004760,1.0,1,0,0,0,0,0,0,0
37,-0.385840,-1.127549,-1.292254,-1.546690,-0.939458,-1.283661,-0.092632,-0.017029,-0.960143,-0.666885,-1.037722,-0.061931,-1.047358,-0.807163,1.927783,1.390994,-1.039013,-0.688075,-0.631744,1.398766,-1.009547,-1.355281,0.0,0,0,0,0,0,0,1,0
431,-1.017290,1.488142,-1.292254,-0.642766,0.167575,-1.066434,0.300422,-0.703620,0.345922,0.942739,-0.989705,-1.227779,-1.047358,-1.014135,0.855506,-0.613589,1.024396,1.629390,-0.858187,-1.216855,-1.009547,1.515955,0.0,0,0,0,0,0,0,0,1


## 7. Guardar artefactos para el notebook de modelado

In [15]:
os.makedirs("artefactos", exist_ok=True)

# --- Clasificador de gastos (texto) ---
joblib.dump(vectorizador_tfidf, "artefactos/tfidf_vectorizer.pkl")
pd.DataFrame({"descripcion_limpia": X_train_txt, "categoria": y_train_cat}).to_csv(
    "artefactos/train_transacciones.csv", index=False
)
pd.DataFrame({"descripcion_limpia": X_test_txt, "categoria": y_test_cat}).to_csv(
    "artefactos/test_transacciones.csv", index=False
)

# --- Clasificador de perfil financiero ---
joblib.dump(scaler_perfil, "artefactos/scaler_perfil.pkl")
joblib.dump(encoder_ahorro, "artefactos/encoder_frecuencia_ahorro.pkl")

X_train_perfil.assign(perfil_financiero=y_train_perfil.values).to_csv(
    "artefactos/train_usuarios.csv", index=False
)
X_test_perfil.assign(perfil_financiero=y_test_perfil.values).to_csv(
    "artefactos/test_usuarios.csv", index=False
)
X_train_perfil_scaled.assign(perfil_financiero=y_train_perfil.values).to_csv(
    "artefactos/train_usuarios_escalado.csv", index=False
)
X_test_perfil_scaled.assign(perfil_financiero=y_test_perfil.values).to_csv(
    "artefactos/test_usuarios_escalado.csv", index=False
)

with open("artefactos/columnas_features_perfil.json", "w", encoding="utf-8") as f:
    json.dump(columnas_features, f, ensure_ascii=False, indent=2)

print("Artefactos guardados en la carpeta 'artefactos/':")
for archivo in sorted(os.listdir("artefactos")):
    print(" -", archivo)

Artefactos guardados en la carpeta 'artefactos/':
 - columnas_features_perfil.json
 - encoder_frecuencia_ahorro.pkl
 - scaler_perfil.pkl
 - test_transacciones.csv
 - test_usuarios.csv
 - test_usuarios_escalado.csv
 - tfidf_vectorizer.pkl
 - train_transacciones.csv
 - train_usuarios.csv
 - train_usuarios_escalado.csv


## Resumen y proximos pasos

**Limpieza de texto**
- Se limpiaron las 13,633 descripciones de transacciones (minusculas, sin tildes, sin codigos de referencia/comercio, sin nombres de ciudad). Ninguna quedo vacia tras la limpieza (0 casos).
- El generador ahora incluye ambiguedad de texto realista: ~12% de las transacciones usan una descripcion generica (COMPRA, PAGO, TRANSFERENCIA, etc.) que no revela la categoria por si sola, y ~10% de las descripciones con comercio incluyen un error tipografico simulado. Esto es intencional: sin esta ambiguedad, el vocabulario queda perfectamente separado por categoria y el clasificador de gastos (Paso 5) da un accuracy irrealmente alto (100%).
- Se documento una colision real entre el comercio "Farmacias Guadalajara" y la ciudad "Guadalajara": afecta a 125 transacciones (0.92% del total). Queda como mejora futura pasar a un diccionario de comercios en vez de una lista generica de ciudades para evitar este tipo de colisiones.

**Tratamiento de outliers**
- Se aplico deteccion y capping por IQR dentro de cada categoria. En este dataset sintetico no se detectaron outliers (la suma total de `valor` no cambio tras el capping), ya que los rangos de valores estan acotados por diseno. El pipeline queda listo para tratar anomalias si en el futuro se usan datos reales.

**Clasificador de gastos (texto)**
- Vocabulario TF-IDF (unigramas + bigramas, `min_df=2`) de 391 terminos (crecio frente a la version anterior por las descripciones genericas y los typos, que generan tokens adicionales), suficiente para diferenciar las 8 categorias sin ser trivialmente separable.
- Split train/test estratificado por categoria: 10,906 transacciones de entrenamiento / 2,727 de prueba, sin fuga de informacion (TF-IDF ajustado solo en train).

**Clasificador de perfil financiero**
- Se agregaron proporciones de gasto por categoria, gasto promedio por transaccion, categoria de gasto dominante (one-hot) y codificacion ordinal de frecuencia de ahorro, totalizando 31 columnas de features.
- Split train/test estratificado por perfil financiero: 640 usuarios de entrenamiento / 160 de prueba, con version sin escalar (para modelos de arbol) y version escalada (para modelos lineales/basados en distancia).

**Artefactos guardados** en `artefactos/` para el siguiente notebook (entrenamiento de modelos): vectorizador TF-IDF, scaler, encoder de frecuencia de ahorro, splits de train/test de ambos datasets, y la lista de columnas de features del modelo de perfil.

**Siguiente paso:** entrenar y evaluar el clasificador de gastos (Paso 5) y el clasificador de perfil financiero (Paso 6), usando estos artefactos.